# Run Horizon + score every injected table

Set **`DATABASE`** to a dataset folder under `datasets/` (e.g. `hospital_2k`).
The notebook runs the real Horizon repair pipeline on **every** injected table in
`datasets/{DATABASE}/injected/` and scores each one against `clean.csv` with the
§6.1 effectiveness metrics (precision / recall / F1), returning one stats row per
table.

Pipeline per table: `load_fds` → `get_ordered_fds` → `FDPatternGraph` →
`load_data` → `repair_dirty_data` → `evaluate_repair`. The FDs and their traversal
order depend only on the schema, so they are computed once; the pattern graph and
the repair are rebuilt per injected table (their values differ).

**Inputs → Outputs:** `datasets/{DATABASE}/{clean.csv, fds.csv, injected/*_r*.csv}`
→ a `stats` table with one row per injected table (P / R / F1, counts, runtime);
repaired tables are written to `output/`.

**Config knobs:**
- `DATABASE` — dataset folder under `datasets/`.

## Setup — imports & sys.path

In [ ]:
import importlib.util
import logging
import sys
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless: get_ordered_fds renders FD/SCC PNGs, no GUI backend
import polars as pl

# horizon/ pipeline modules use flat imports and need horizon/ on sys.path; eval
# uses package imports (horizon.*) and needs the repo root. Put both on the path,
# repo root first, and load horizon/horizon.py by file path under a private name to
# dodge the package/module name clash on `horizon`.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

spec = importlib.util.spec_from_file_location("horizon_pipeline", HZN / "horizon.py")
pipe = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipe)

from fd_pattern_graph import FDPatternGraph
from static_fd_analysis import get_ordered_fds
from horizon.utils.loaders import load_fds, load_table

from eval.effectiveness_eval import evaluate_repair

DATASETS = REPO / "datasets"
OUTPUT = REPO / "output"
OUTPUT.mkdir(exist_ok=True)

logging.getLogger("horizon").setLevel(logging.WARNING)  # mute per-tuple progress logs

## Configuration

In [ ]:
# --- config: the only knob is the dataset folder under datasets/ ---
# swap to any folder there, e.g. "insurance_claims_2k"
DATABASE = "hospital_2k"

## Run Horizon on every injected table + score

In [ ]:
def parse_stem(stem):
    """injected stem -> (etype, rate, level): 'e1_r05_high' -> ('e1', 5, 'high')."""
    etype, _, tail = stem.partition("_r")
    rate_str, _, level = tail.partition("_")
    return etype, int(rate_str), (level or "")


ds_dir = DATASETS / DATABASE
clean_path = ds_dir / "clean.csv"
injected = sorted((ds_dir / "injected").glob("*_r*.csv"))
assert injected, f"no injected/*_r*.csv under {ds_dir}"

# FDs + traversal order depend only on the schema -> compute once
set_of_fds = load_fds(ds_dir, clean_path)
ordered_fds = get_ordered_fds(set_of_fds, DATABASE, OUTPUT)[0]

# clean ground truth, columns normalized the way the pipeline normalizes the data
clean = pl.read_csv(clean_path, infer_schema_length=0)
clean = clean.rename({c: c.strip().lower() for c in clean.columns})

rows = []
for path in injected:
    etype, rate, level = parse_stem(path.stem)
    t0 = time.perf_counter()
    graph = FDPatternGraph(path, set_of_fds)
    dirty = load_table(path)
    cleaned_out = OUTPUT / f"{DATABASE}_cleaned_{path.stem}.csv"
    # repair_dirty_data sinks the repaired table to cleaned_out; load it back to score
    pipe.repair_dirty_data(path, cleaned_out, ordered_fds, graph.repair_table, graph)
    elapsed = time.perf_counter() - t0
    cleaned = load_table(cleaned_out)

    m = evaluate_repair(clean, dirty, cleaned)
    rows.append(
        {
            "table": path.stem,
            "etype": etype,
            "rate": rate,
            "level": level,
            "n_dirty": m["n_dirty"],
            "n_repaired": m["n_repaired"],
            "n_correct": m["n_correct"],
            "precision": round(m["precision"], 4),
            "recall": round(m["recall"], 4),
            "f1": round(m["f1"], 4),
            "secs": round(elapsed, 2),
        }
    )
    print(f"{path.stem:24s}  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}")

stats = pl.DataFrame(rows).sort(["etype", "rate", "level"])
with pl.Config(tbl_rows=len(rows)):
    display(stats)